# 🚗 DPR-RIA: Dual-Path Road Infrastructure Analytics — GPU Training

**Run this notebook on Google Colab (T4/A100) or Kaggle (P100/T4) for fastest training.**

This notebook trains the full custom architecture:
- **SPD-Conv** backbone — preserves distant pothole pixels
- **BMS-SPPF** — bidirectional pooling for cracks + potholes  
- **P2/P3/P4/P5** heads — 40 m range detection

All custom modules are defined inline — no file uploads needed beyond the dataset.

---
### Setup options:
| Platform | GPU | Expected train time (100 ep) |
|---|---|---|
| Google Colab Free | T4 16 GB | ~8-10 hrs |
| Google Colab Pro | A100 40 GB | ~2-3 hrs |
| Kaggle | P100 16 GB | ~8-10 hrs |

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics sahi geojson filterpy grad-cam transformers huggingface_hub
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2 — Dataset Setup

### Option A: Google Drive (recommended for Colab)
Upload your dataset zip to Google Drive, then mount it.

### Option B: Kaggle Dataset  
Add the RDD2022 dataset from Kaggle directly (no upload needed).

### Dataset structure expected:
```
datasets/
  train/images/   train/labels/
  valid/images/   valid/labels/
  test/images/    test/labels/
```

In [ ]:
import os

# ── Option A: Mount Google Drive ───────────────────────────────────────────────
USE_GDRIVE = True  # Set False if using Kaggle or direct upload

if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # ↓ Change this to the path of your dataset zip on Drive
    GDRIVE_ZIP = '/content/drive/MyDrive/RDD2022_datasets.zip'
    if os.path.exists(GDRIVE_ZIP):
        !unzip -q {GDRIVE_ZIP} -d /content/
        DATASET_ROOT = '/content/datasets'
    else:
        # If already extracted, just point to the folder
        DATASET_ROOT = '/content/drive/MyDrive/RoadDamageDetection/datasets'
else:
    # Option B: Kaggle — dataset is already at /kaggle/input/
    # Adjust to match your Kaggle dataset path
    DATASET_ROOT = '/kaggle/input/rdd2022/datasets'

# Verify structure
for split in ['train', 'valid', 'test']:
    imgs = len(os.listdir(f'{DATASET_ROOT}/{split}/images'))
    lbls = len(os.listdir(f'{DATASET_ROOT}/{split}/labels'))
    print(f'{split:6s}: {imgs:6d} images  {lbls:6d} labels')

## Cell 3 — Write Custom Module Files

In [ ]:
import os
os.makedirs('/content/src/models', exist_ok=True)
os.makedirs('/content/src/utils', exist_ok=True)

In [ ]:
%%writefile /content/src/models/spd_conv.py
import torch
import torch.nn as nn
from ultralytics.nn.modules.conv import Conv

class SPDConv(nn.Module):
    """
    Space-to-Depth Convolution.
    Replaces strided convolutions — preserves pixel details of distant potholes.
    Instead of discarding pixels by striding, it shuffles spatial info into channels.
    """
    def __init__(self, c1, c2, k=3, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.scale = 2
        c_spd = c1 * (self.scale ** 2)   # channels expand by scale^2
        self.conv = Conv(c_spd, c2, k, s=1, p=p, g=g, d=d, act=act)

    def forward(self, x):
        B, C, H, W = x.shape
        s = self.scale
        if H % s != 0 or W % s != 0:
            x = nn.functional.pad(x, (0, (s - W % s) % s, 0, (s - H % s) % s))
        sub = [x[:, :, i::s, j::s] for i in range(s) for j in range(s)]
        return self.conv(torch.cat(sub, dim=1))

In [ ]:
%%writefile /content/src/models/bms_sppf.py
import torch
import torch.nn as nn
from ultralytics.nn.modules.conv import Conv

class BMS_SPPF(nn.Module):
    """
    Bidirectional Multi-Scale SPPF.
    Standard SPPF uses only cascaded max-pool (one scale).
    BMS-SPPF adds an avg→max branch to simultaneously capture:
      - Elongated crack gradients (avg-pool smoothing)
      - Sharp pothole edge peaks (max-pool)
    Signature mirrors SPPF(c1, c2, k) so YOLO YAML parser works correctly.
    """
    def __init__(self, c1: int, c2: int, k: int = 5) -> None:
        super().__init__()
        c_ = c1 // 2
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c_ * 5, c2, 1, 1)   # 5 branches concatenated
        self.m   = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)
        self.avg = nn.AvgPool2d(kernel_size=k, stride=1, padding=k // 2)
        self.m2  = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x  = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        y_bidi = self.m2(self.avg(x))   # crack-aware bidirectional branch
        return self.cv2(torch.cat([x, y1, y2, y3, y_bidi], dim=1))

In [ ]:
%%writefile /content/src/models/yolo11-spd-bmssppf-p2.yaml
# DPR-RIA: YOLOv11 with SPD-Conv + BMS-SPPF + P2 small-object head
# SPD-Conv (Focus proxy): no information loss during downsampling
# BMS-SPPF (SPPF proxy):  bidirectional pooling for cracks + potholes
# P2 head (160x160):      captures potholes at 15-40 m range

nc: 4
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, Focus,  [64]]            # 0-P1/2   SPD-Conv proxy
  - [-1, 1, Focus,  [128]]           # 1-P2/4   SPD-Conv proxy
  - [-1, 2, C3k2,   [256, False, 0.25]]   # 2
  - [-1, 1, Focus,  [256]]           # 3-P3/8   SPD-Conv proxy
  - [-1, 2, C3k2,   [512, False, 0.25]]   # 4
  - [-1, 1, Conv,   [512, 3, 2]]     # 5-P4/16
  - [-1, 2, C3k2,   [512, True]]     # 6
  - [-1, 1, Conv,   [1024, 3, 2]]    # 7-P5/32
  - [-1, 2, C3k2,   [1024, True]]    # 8
  - [-1, 1, SPPF,   [1024, 5]]       # 9  BMS-SPPF proxy
  - [-1, 2, C2PSA,  [1024]]          # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 11
  - [[-1, 6], 1, Concat, [1]]                   # 12 cat P4
  - [-1, 2, C3k2, [512, False]]                 # 13
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 14
  - [[-1, 4], 1, Concat, [1]]                   # 15 cat P3
  - [-1, 2, C3k2, [256, False]]                 # 16 P3/8
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 17
  - [[-1, 2], 1, Concat, [1]]                   # 18 cat P2
  - [-1, 2, C3k2, [128, False]]                 # 19 P2/4-tiny (40m range)
  - [-1, 1, Conv, [128, 3, 2]]                  # 20
  - [[-1, 16], 1, Concat, [1]]                  # 21 cat P3 head
  - [-1, 2, C3k2, [256, False]]                 # 22 P3/8
  - [-1, 1, Conv, [256, 3, 2]]                  # 23
  - [[-1, 13], 1, Concat, [1]]                  # 24 cat P4 head
  - [-1, 2, C3k2, [512, False]]                 # 25 P4/16
  - [-1, 1, Conv, [512, 3, 2]]                  # 26
  - [[-1, 10], 1, Concat, [1]]                  # 27 cat P5 head
  - [-1, 2, C3k2, [1024, True]]                 # 28 P5/32
  - [[19, 22, 25, 28], 1, Detect, [nc]]         # Detect(P2,P3,P4,P5)

## Cell 4 — Register Custom Modules & Write data.yaml

In [ ]:
import sys
sys.path.insert(0, '/content')

# Register BEFORE importing YOLO so parse_model sees the patches
import ultralytics.nn.modules as _modules
import ultralytics.nn.tasks  as _tasks

from src.models.spd_conv import SPDConv
from src.models.bms_sppf import BMS_SPPF

# SPD-Conv → Focus proxy (Focus is in base_modules → parser injects c1, c2)
setattr(_modules, 'Focus', SPDConv)
setattr(_tasks,   'Focus', SPDConv)

# BMS-SPPF → SPPF proxy (SPPF is in base_modules → parser injects c1, c2, k)
setattr(_modules, 'SPPF', BMS_SPPF)
setattr(_tasks,   'SPPF', BMS_SPPF)

from ultralytics import YOLO

# Build model
model = YOLO('/content/src/models/yolo11-spd-bmssppf-p2.yaml')
model.load('yolo11n.pt')   # warm-start with ImageNet backbone
model.info()
print("\n✅ Custom model built: SPDConv + BMS-SPPF + P2/P3/P4/P5 heads")

In [ ]:
import yaml

data_config = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 4,
    'names': {
        0: 'D00_Longitudinal_Crack',
        1: 'D10_Transverse_Crack',
        2: 'D20_Alligator_Crack',
        3: 'D40_Pothole'
    }
}

DATA_YAML = '/content/rdd2022.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f)
print(f"✅ data.yaml written → {DATA_YAML}")
print(yaml.dump(data_config))

## Cell 5 — Train! 🚀

> **Recommended:** Use `epochs=100`, `batch=16` on T4/P100, `batch=32` on A100.
> Set `QUICK_TEST=True` to run just 3 epochs as a sanity check first.

In [ ]:
QUICK_TEST = False  # ← Set True for a 3-epoch sanity check first

BATCH  = 16   # Increase to 32 on A100 for faster training
EPOCHS = 3 if QUICK_TEST else 100

results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = 640,
    batch       = BATCH,
    device      = 0,           # GPU 0
    workers     = 4,
    # ── Blueprint augmentations ─────────────────────────────────────────────
    mosaic      = 1.0,         # Mosaic: exposes model to small/distant objects
    perspective = 0.001,       # Simulates 40 m viewing angle
    degrees     = 10.0,
    translate   = 0.1,
    scale       = 0.5,
    fliplr      = 0.5,
    flipud      = 0.0,
    # ── NMS settings (handles P2's extra anchors) ──────────────────────────
    conf        = 0.15,        # Filter low-conf anchors BEFORE NMS
    iou         = 0.5,
    # ── Output ──────────────────────────────────────────────────────────────
    project     = '/content/runs',
    name        = 'dual_path_rdd2022',
    exist_ok    = True,
    patience    = 20,
    save        = True,
    save_period = 5,
    plots       = True,
)

print("\n" + "="*60)
print("✅ Training complete!")
print(f"   mAP50:    {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"   mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"   Best weights: /content/runs/dual_path_rdd2022/weights/best.pt")

## Cell 6 — Validate & Inspect Results

In [ ]:
from ultralytics import YOLO as _YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

best_weights = '/content/runs/dual_path_rdd2022/weights/best.pt'

# Re-register modules before loading trained model
setattr(_modules, 'Focus', SPDConv)
setattr(_tasks,   'Focus', SPDConv)
setattr(_modules, 'SPPF',  BMS_SPPF)
setattr(_tasks,   'SPPF',  BMS_SPPF)

trained_model = _YOLO(best_weights)

# Run validation on test set
val_results = trained_model.val(
    data=DATA_YAML,
    split='test',
    conf=0.25,
    iou=0.5,
    plots=True,
)

print(f"\nTest Set Results:")
print(f"  mAP50:    {val_results.box.map50:.4f}")
print(f"  mAP50-95: {val_results.box.map:.4f}")
print(f"  Per-class mAP50: {val_results.box.maps}")

# Show training plots
run_dir = Path('/content/runs/dual_path_rdd2022')
for plot_file in ['results.png', 'confusion_matrix.png', 'labels.jpg']:
    fpath = run_dir / plot_file
    if fpath.exists():
        img = mpimg.imread(str(fpath))
        plt.figure(figsize=(16, 8))
        plt.imshow(img)
        plt.title(plot_file)
        plt.axis('off')
        plt.show()

## Cell 7 — Run Inference on a Sample Image

In [ ]:
import os, random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np

CLASS_NAMES = ['D00_Longitudinal', 'D10_Transverse', 'D20_Alligator', 'D40_Pothole']
CLASS_COLORS = ['#00ff88', '#ffaa00', '#ff4444', '#aa00ff']

# Pick a random test image
test_dir = os.path.join(DATASET_ROOT, 'test/images')
sample_img = os.path.join(test_dir, random.choice(os.listdir(test_dir)))

results_infer = trained_model.predict(
    source=sample_img,
    conf=0.25,
    iou=0.5,
    imgsz=640,
    save=False,
)

# Visualise
img = np.array(Image.open(sample_img))
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(img)

for r in results_infer:
    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        color = CLASS_COLORS[cls]
        rect = mpatches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{CLASS_NAMES[cls]} {conf:.2f}",
                color=color, fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))

ax.set_title(f"Detections: {os.path.basename(sample_img)}", fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()
print(f"Detected {sum(len(r.boxes) for r in results_infer)} objects")

## Cell 8 — Save Best Weights to Google Drive
So you can download and use them locally.

In [ ]:
import shutil

# ── Copy best.pt to Google Drive ──────────────────────────────────────────────
SAVE_DEST = '/content/drive/MyDrive/RoadDamageDetection/'
os.makedirs(SAVE_DEST, exist_ok=True)

src  = '/content/runs/dual_path_rdd2022/weights/best.pt'
dest = os.path.join(SAVE_DEST, 'dual_path_rdd2022_best.pt')
shutil.copy(src, dest)
print(f"✅ Saved best.pt → {dest}")

# Also save the full results folder
results_dest = os.path.join(SAVE_DEST, 'runs_dual_path_rdd2022')
if not os.path.exists(results_dest):
    shutil.copytree('/content/runs/dual_path_rdd2022', results_dest)
    print(f"✅ Results folder saved → {results_dest}")

## Cell 9 — Use the Trained Model Locally

Once training completes, download `dual_path_rdd2022_best.pt` from Drive, copy it to your Mac, and run locally:

```bash
# On your Mac
cd /Users/vjkiran/Development/RoadDamageDetector
source venv/bin/activate

# Run SAHI auditor inference with the GPU-trained model
python3 src/data/sahi_inference.py \
    --weights dual_path_rdd2022_best.pt \
    --image /path/to/road_image.jpg

# Export to TensorRT for Jetson Orin
python3 src/scripts/export_tensorrt.py \
    --weights dual_path_rdd2022_best.pt \
    --int8 --data src/data/rdd2022.yaml
```